In [1]:
# R2_CAR171_Canopus.ipynb
# Estimating the stray-light from Canopus star for the Commissioning Activity Request (CAR) 171. 
# This notebook is based on the R1_Rosalia_stray_example.ipynb notebook, but it is adapted to the specific case of Canopus.
# - Alejandro S. Borlaff / NASA Ames / a.s.borlaff@nasa.gov. March 11, 2026.

import sys
sys.path.append("/Users/aborlaff/NASA/ROSALIA/")

import rosalia as rs
import numpy as np

print(rs.__file__)

The archive is unstable and may perform below expectations. Please avoid launching intense Python query showers. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the recent infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive
/Users/aborlaff/NASA/ROSALIA/rosalia/__init__.py


In [2]:

def make_stray_plot(input_name, ext, fe2mu=True, vmin=None, vmax=None, color_label = 'Surface brightness (mag arcsec$^{-2}$)'):
    import matplotlib.pyplot as plt
    from astropy.utils.data import get_pkg_data_filename
    from astropy.wcs import WCS as astropy_wcs
    from astropy.io import fits
    import os
    plt.style.use(os.path.dirname(rs.__file__) + "/style/nature_style.mplstyle")

    hdu = fits.open(input_name)
    data = hdu[ext].data
    wcs = astropy_wcs(hdu[ext].header)

    fig, ax = plt.subplots(figsize=(7, 6), subplot_kw=dict(projection=wcs))

    if fe2mu:
        data = rs.detectors.fe2mu(data, telescope="Roman", instrument="WFI", filter_name="F129")
        
    data[np.isinf(data)] = np.nan
    data[data == 0] = np.nan

    vmin = np.nanpercentile(data, 5)
    vmax = np.nanpercentile(data, 95)
    
    print(vmin)
    print(vmax)
    im=ax.imshow(data, vmin=vmin, vmax=vmax, origin='lower', cmap="RdYlBu")
    ax.set(xlabel='Right ascension (degrees)', ylabel='Declination (degrees)')
    cbar = plt.colorbar(im, ax=ax, location='right', )
    cbar.set_label(label=color_label,weight='bold')
    plt.savefig(input_name.replace(".fits", ".png"), dpi=300)
    plt.show()


In [3]:
# Let's find Canopus (alpha Carinae), the star selected for CAR171. 
# This star is in the continuous viewing zone of Roman, and it is the second brightest star in the sky.
import astropy.units as u
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord

alfCar = Simbad.query_object('Canopus')
ra_star = alfCar["ra"][0]
dec_star = alfCar["dec"][0]

target = SkyCoord(ra_star*u.deg, dec_star*u.deg, frame="icrs")
print(target)


<SkyCoord (ICRS): (ra, dec) in deg
    (95.98795783, -52.69566138)>


In [4]:
# These are the coordinates of specially sensitive locations 
# for stray-light, measured in offset degrees from the center of WFI focal plane
dX = np.array([-3.7, 3.7, 6.05, 2.45, -2.45, -6.05, -2, 2, -1.3, 1.3, -0.0688, 0.0688, -0.138, 0.138])
dY = np.array([5.27, 5.27, 1.18, -5.07, -5.07, 1.18, 6.15, 6.15, 1, 1, -0.0312, -0.0312, -0.0035, -0.0048])
number_of_points_of_interest = len(dX)
# To measure the stray-light from those sources, we need to place the center 
# of Roman / WFI at a certain distance and position angle from the source 
# that generates the stray-light. We will name that source the "offending" source. 

# ROSALIA has a specific tool to compute those locations. 

# The optimal position angle of Roman depends with time. 
# Let's set an approximate time for the Commissioning Activities
from astropy.time import Time
date = Time('2026-11-21T00:00:00.0', format='isot', scale='utc')

ra_wfi = np.zeros(number_of_points_of_interest)
dec_wfi = np.zeros(number_of_points_of_interest)
pa_wfi = np.zeros(number_of_points_of_interest)

for i in range(number_of_points_of_interest):
    offset_pointing = rs.telescopes.Roman.find_wfi_center_for_offset_target(ra_target=target.ra.degree,
                                                      dec_target=target.dec.degree,
                                                      mjd=date.mjd,
                                                      dX=dX[i], dY=dY[i])
    ra_wfi[i] = offset_pointing["ra_wficen"]
    dec_wfi[i] = offset_pointing["dec_wficen"]
    pa_wfi[i] = offset_pointing["pa_wfi"]


In [5]:
if True:
    i = 0
    ra = ra_wfi[i]
    dec = dec_wfi[i]
    pa = pa_wfi[i]
    mjd = date.mjd
    bandpass="F129"
    exptime=600
    rosalia_stray = rs.correct.rosalia_stray(ra=ra, dec=dec, 
                                             PA=pa, date=date, bandpass=bandpass, 
                                             exptime=exptime, radius=1,
                                             g_mag_max=15)

/Users/aborlaff/NASA/ROSALIA/notebooks/WFI_F129_RA_106.624_DEC_-53.595_MJD_61365.00000_PA_-121.23.fits














INFO: radius parameter (minimum distance to search for individual stars) is > 0.5 degrees.
Gaia/2MASS/WISE query database can take several minutes to process. Please be patient.
Querying Gaia/2MASS/WISE/JPL Horizons databases. This might take a few minutes... ⢿

2026-03-13 15:33:13 INFO     Retrieving tables...


INFO: Retrieving tables... [astroquery.utils.tap.core]
Querying Gaia/2MASS/WISE/JPL Horizons databases. This might take a few minutes... ⣯

2026-03-13 15:33:15 INFO     Parsing tables...


INFO: Parsing tables... [astroquery.utils.tap.core]
Querying Gaia/2MASS/WISE/JPL Horizons databases. This might take a few minutes... ⣾

...

18it [06:05, 20.30s/it]


SCA 18 - Subarray X=17.92 - Subarray Y=17.92
SCA 18 out of 18: 98.44% completed


18it [00:00, 420.75it/s]
18it [00:00, 423.49it/s]


Output saved in: /Users/aborlaff/NASA/ROSALIA/notebooks/WFI_F129_RA_106.624_DEC_-53.595_MJD_61365.00000_PA_-121.23_stray.fits




























KeyError: Header keyword not found
KeyError: Header keyword not found
KeyError: Header keyword not found
KeyError: Header keyword not found


TypeError: 'list' object cannot be interpreted as an integer

In [ ]:
scale_drz_name = "/Users/aborlaff/NASA/ROSALIA/notebooks/WFI_F129_RA_106.624_DEC_-53.595_MJD_61365.00000_PA_-121.23_stray_drz_scaled.fits"
mainoff_drz_name = "/Users/aborlaff/NASA/ROSALIA/notebooks/WFI_F129_RA_106.624_DEC_-53.595_MJD_61365.00000_PA_-121.23_stray_main_off_drz_scaled.fits"
ext = 1

make_stray_plot(input_name=scale_drz_name, ext=ext, fe2mu=True, vmin=None, vmax=None)
make_stray_plot(input_name=mainoff_drz_name, ext=ext, fe2mu=False, vmin=None, vmax=None, 
                color_label = "Main offending source (ID)")



In [ ]:
for i in range(number_of_points_of_interest):
    ra = ra_wfi[i]
    dec = dec_wfi[i]
    pa = pa_wfi[i]
    mjd = mjd
    bandpass="F129"
    exptime=600
    rosalia_stray = rs.correct.rosalia_stray(ra=ra, dec=dec, 
                                             PA=pa, date=date, bandpass=bandpass, 
                                             exptime=exptime, radius=1,
                                             g_mag_max=15)

In [ ]:
rosalia_stray